# KineticsRecon — Google Colab Training

Kinetics-Preserving Accelerated Breast DCE-MRI Reconstruction for pCR Prediction

**Stage 2 only** (MAMAMIA fine-tune). Requires DUKE subset (~7.2 GB) in Google Drive.

> Set Runtime → Change runtime type → **A100 GPU** before running.

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    raise RuntimeError('No GPU — change runtime type to A100 and restart')

In [ ]:
# ── Cell 2: Mount Drive & set paths ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

MAMAMIA_ROOT  = '/content/drive/MyDrive/colab_data'  # folder you uploaded
MAMAMIA_CSV   = f'{MAMAMIA_ROOT}/clinical_and_imaging_info.xlsx'
FINETUNE_DIR  = '/content/runs/finetune'
RESULTS_DIR   = '/content/results'

os.makedirs(FINETUNE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Quick sanity check
assert pathlib.Path(MAMAMIA_CSV).exists(), f'Clinical CSV not found at {MAMAMIA_CSV}'
cases = [d.name for d in pathlib.Path(f'{MAMAMIA_ROOT}/images').iterdir() if d.is_dir()]
print(f'Found {len(cases)} cases in {MAMAMIA_ROOT}/images')

In [ ]:
# ── Cell 3: Clone repo & install deps ───────────────────────────────────────
import subprocess, sys

REPO = 'https://github.com/sathvikloke/KineticsRecon.git'
REPO_DIR = '/content/KineticsRecon'

if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Install complete.')

In [ ]:
# ── Cell 4: Stage 2 — Fine-tune on MAMAMIA/DUKE ─────────────────────────────
# Trains from scratch (no Stage-1 checkpoint needed).  ~1.5h on A100.
!python train.py \
  --stage        finetune \
  --mamamia_root {MAMAMIA_ROOT} \
  --mamamia_csv  {MAMAMIA_CSV} \
  --output_dir   {FINETUNE_DIR} \
  --epochs       100 \
  --batch_size   8 \
  --lr           1e-4 \
  --acceleration 6 \
  --n_phases     3 \
  --freeze_epochs 0 \
  --w_img   1.0 \
  --w_pk    0.5 \
  --w_curve 0.3 \
  --w_pcr   1.0 \
  --amp \
  --num_workers 2

In [ ]:
# ── Cell 5: Verify checkpoint ────────────────────────────────────────────────
import torch, json

ckpt_path = f'{FINETUNE_DIR}/best.pt'
assert pathlib.Path(ckpt_path).exists(), f'Checkpoint not found: {ckpt_path}'

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
print(f"Best epoch : {ckpt['epoch']}")
print(f"Best AUC   : {ckpt['best_metric']:.4f}")

# Print training history summary
hist_path = pathlib.Path(f'{FINETUNE_DIR}/history.json')
if hist_path.exists():
    history = json.load(open(hist_path))
    best = max(history, key=lambda x: x['auc'])
    final = history[-1]
    print(f"Best  epoch {best['epoch']:3d}: AUC={best['auc']:.4f}  SSIM={best['ssim']:.4f}  PSNR={best['psnr']:.2f}")
    print(f"Final epoch {final['epoch']:3d}: AUC={final['auc']:.4f}  SSIM={final['ssim']:.4f}  PSNR={final['psnr']:.2f}")

In [ ]:
# ── Cell 6: Evaluate on test split ───────────────────────────────────────────
# Runs the full evaluation: SSIM, PSNR, NMSE, curve RMSE, pCR AUC/Sens/Spec
!python -m evaluation.metrics \
  --checkpoint   {FINETUNE_DIR}/best.pt \
  --mamamia_root {MAMAMIA_ROOT} \
  --mamamia_csv  {MAMAMIA_CSV} \
  --acceleration 6 \
  --n_phases     3 \
  --batch_size   4 \
  --out_csv      {RESULTS_DIR}/test_R6.csv

print('\n--- Case-level predictions ---')
import csv
with open(f'{RESULTS_DIR}/test_R6.csv') as f:
    for row in csv.DictReader(f):
        print(f"  {row['case_id']}  prob={float(row['pcr_prob']):.3f}  label={row['pcr_label']}")

In [ ]:
# ── Cell 7: Training curves ───────────────────────────────────────────────────
import json, matplotlib.pyplot as plt

history = json.load(open(f'{FINETUNE_DIR}/history.json'))
epochs  = [h['epoch'] for h in history]
aucs    = [h['auc']   for h in history]
ssims   = [h['ssim']  for h in history]
psnrs   = [h['psnr']  for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, aucs,  'b-o', markersize=3, lw=1.5)
axes[0].axhline(max(aucs), color='b', linestyle='--', alpha=0.5,
                label=f'Best={max(aucs):.4f} @ ep{epochs[aucs.index(max(aucs))]}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('pCR AUC')
axes[0].set_title('pCR AUC (val)'); axes[0].legend(); axes[0].set_ylim(0, 1.05)

axes[1].plot(epochs, ssims, 'g-o', markersize=3, lw=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('SSIM')
axes[1].set_title('SSIM (val)')

axes[2].plot(epochs, psnrs, 'r-o', markersize=3, lw=1.5)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('PSNR (dB)')
axes[2].set_title('PSNR (val)')

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150)
plt.show()
print(f'Saved to {RESULTS_DIR}/training_curves.png')

In [ ]:
# ── Cell 8: Ablation — R = 4×, 6×, 8× ───────────────────────────────────────
# Re-evaluates the same checkpoint at different acceleration factors.
# NOTE: The checkpoint was trained at R=6, so R=4 will be generous
#       and R=8 will be challenging — exactly the pattern papers show.

import json, pathlib

ablation_results = {}

for R in [4, 6, 8]:
    out_csv = f'{RESULTS_DIR}/test_R{R}.csv'
    !python -m evaluation.metrics \
      --checkpoint   {FINETUNE_DIR}/best.pt \
      --mamamia_root {MAMAMIA_ROOT} \
      --mamamia_csv  {MAMAMIA_CSV} \
      --acceleration {R} \
      --n_phases     3 \
      --batch_size   4 \
      --out_csv      {out_csv}
    summary = json.load(open(out_csv.replace('.csv', '.json')))
    ablation_results[R] = summary
    print(f'R={R}×  SSIM={summary["SSIM"]:.4f}  PSNR={summary["PSNR"]:.2f}  AUC={summary["pCR_AUC"]:.4f}')

# Pretty table for paper
print('\n=== Table for paper ===')
print(f'{"R":>4}  {"SSIM":>6}  {"PSNR":>7}  {"NMSE":>8}  {"curveRMSE":>10}  {"AUC":>6}  {"Sens":>6}  {"Spec":>6}')
for R, r in ablation_results.items():
    print(f'{R:>4}x  {r["SSIM"]:6.4f}  {r["PSNR"]:7.2f}  {r["NMSE"]:8.4f}  {r["curve_RMSE"]:10.4f}  '
          f'{r["pCR_AUC"]:6.4f}  {r["pCR_sens"]:6.3f}  {r["pCR_spec"]:6.3f}')

In [ ]:
# ── Cell 9: Save results & checkpoint to Google Drive ────────────────────────
import shutil, pathlib

DRIVE_RESULTS = '/content/drive/MyDrive/KineticsRecon_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Copy checkpoint
shutil.copy(f'{FINETUNE_DIR}/best.pt',       f'{DRIVE_RESULTS}/best.pt')
shutil.copy(f'{FINETUNE_DIR}/history.json',  f'{DRIVE_RESULTS}/history.json')

# Copy all result files
for p in pathlib.Path(RESULTS_DIR).glob('*'):
    shutil.copy(p, f'{DRIVE_RESULTS}/{p.name}')

print(f'Saved to {DRIVE_RESULTS}:')
for p in sorted(pathlib.Path(DRIVE_RESULTS).iterdir()):
    print(f'  {p.name}  ({p.stat().st_size//1024} KB)')